# AAM, signed-event decoding, and explicit TS analysis

Search and decoding remain separate. TS analysis consumes a mapping and bond events chosen explicitly from the decoded result. No geometry selector is invoked.


In [ ]:
import numpy as np
import mappa
from mappa import (
    AAMProblem, AAMSearchConfig, MolecularEndpoint,
    ReactionContext, ResolvedMechanism, AtomBijection,
    TransitionStateTarget, VibrationalModes, search_aam, analyze_transition_state,
)
from mappa.postprocessing import decode_events, EventDecodeConfig
from mappa.artifacts import reaction_record
print('mappa', mappa.__version__)


## 1. Construct immutable molecular endpoints

Each endpoint contains elements, Cartesian coordinates, and a complete symmetric WBO matrix. The example changes two bonds and uses distinct elements so its expected mapping is easy to audit.

In [ ]:
elements = ('C', 'N', 'O', 'H')
xyz = np.array([[0.,0.,0.], [1.,0.,0.], [0.,1.,0.], [1.,1.,0.2]])

def wbo_matrix(edges):
    matrix = np.zeros((len(elements), len(elements)))
    for left, right, value in edges:
        matrix[left, right] = matrix[right, left] = value
    return matrix

reactant = MolecularEndpoint(
    elements, xyz, wbo_matrix([(0,1,1.), (2,3,1.)]), label='R')
product = MolecularEndpoint(
    elements, xyz + 0.1, wbo_matrix([(0,2,1.), (1,3,1.)]), label='P')
problem = AAMProblem(reactant, product, name='tutorial_reaction')
print(problem.name, problem.atom_count, 'atoms')

## 2. Search and inspect the retained graph

`branch_limit` bounds live branch admissions per seed/cut, not total history or RAM. `workers` controls cut-sweep processes.


In [ ]:
config = AAMSearchConfig(seed_count=1, branch_limit=100)
aam = search_aam(problem, config, workers=1)
print('states:', len(aam.graph.states))
print('terminal states:', len(aam.graph.terminals))
print('search capped:', aam.graph.capped)


## 3. Decode signed bond events

Completeness applies to the saved families and requested event window. A candidate contains one witness and supports exact family-constrained queries.


In [ ]:
decoded = decode_events(aam, EventDecodeConfig())
assert decoded.complete
print('complete:', decoded.complete)
print('event classes:', len(decoded.candidates))
for candidate in decoded.candidates:
    print(candidate.id, candidate.total, candidate.mapping, candidate.events)


In [ ]:
# This distinct-element example has a single event class. Select it explicitly.
assert len(decoded.candidates) == 1
candidate = decoded.candidates[0]
assert decoded.query(candidate, candidate.mapping).status == 'allowed'
mapping = AtomBijection.from_mapping(candidate.mapping)
# Events use reactant atom indices at this boundary.
broken = tuple(tuple(pair) for pair in candidate.events['broken'])
formed = tuple(tuple(pair) for pair in candidate.events['formed'])
core = tuple(sorted({a for bond in (*broken, *formed) for a in bond}))
reaction = ReactionContext(problem, config, (
    ResolvedMechanism(mapping, broken, formed, core),
))
print('resolved reaction:', reaction_record(reaction))


### Explicit downstream selection

The chosen witness is a family-certified correspondence, not a geometry certificate. Coordinate-orientation filtering, when needed, is an explicit separate call described in `CHIRALITY.md`.


## 4. Mechanism-local TS processing

TS processing performs independent R→TS and P→TS partial AAM searches. Product assignments are pulled into R indexing, exact correlated core tuples are merged, and imaginary modes are scored. These are core assignments—not full-bijection enumeration.

In [ ]:
target_molecule = MolecularEndpoint(
    elements, xyz + 0.05,
    wbo_matrix([(0,1,.5), (2,3,.5), (0,2,.5), (1,3,.5)]),
    label='TS')
modes = np.zeros((1, 4, 3))
modes[0, :, 0] = (1., -1., -1., 1.)
target = TransitionStateTarget(
    target_molecule, VibrationalModes(np.array([-500.]), modes))
ts = analyze_transition_state(reaction, target, search_config=config)
selected = ts.mechanisms[0].selected
print('core assignment:', selected.assignment.as_dict())
print('endpoint support:', sorted(selected.sources))
print('mode/frequency:', selected.mode_index, selected.frequency)
print('score:', round(selected.score, 8))

## 5. CLI and artifacts

The default CLI runs search and event decoding, writing `aam.json`, `search.html`, and `events.json` with completeness evidence.

```bash
mappa --stage events --reactant-npz R.npz --product-npz P.npz \
  --workers 4 --output mapping
```

For TS analysis, serialize the explicitly constructed context with `reaction_record(reaction)` and pass it using `--stage ts --reaction-json selected_reaction.json --target-npz target.npz`.
